# Setup

In [ ]:
import json
import time
import requests
from pathlib import Path
from bs4 import BeautifulSoup
import json
import fitz
import gdown
import tempfile
import os
import re
from google import genai as google_genai

# globais
REPARSE = False
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}
GEMINI_API_KEY_T1 = os.getenv("GEMINI_API_KEY_T1")
DATE_URL_PATTERN = re.compile(r"/(\d{4})/")
MAX_CHARS_INICIO = 500
MAX_CHARS_FIM = 500

google_client = google_genai.Client(api_key=GEMINI_API_KEY_T1)

# carrega dados já processados
with open("../data/raw/pages.json", "r", encoding="utf-8") as f:
    pages_found = json.load(f)

with open("../data/raw/pdfs.json", "r", encoding="utf-8") as f:
    pdfs_found = json.load(f)

# carrega erros de formato já conhecidos
format_errors_path = Path("../data/parsed/pdfs_format_errors.json")
if format_errors_path.exists():
    with open(format_errors_path, "r", encoding="utf-8") as f:
        format_errors = set(json.load(f))
else:
    format_errors = set()

In [ ]:
# funções de extração de data
def extract_date_from_url(url):
    match = DATE_URL_PATTERN.search(url)
    if match:
        return match.group(1)
    return None

def extract_date_from_text(text):
    truncated = text[:MAX_CHARS_INICIO] + "\n...\n" + text[-MAX_CHARS_FIM:]
    prompt = f"Qual é o ano de publicação deste documento? Responda APENAS com o ano no formato YYYY. Se não encontrar, responda exatamente: None\n\n{truncated}"
    for attempt in range(3):
        try:
            response = google_client.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=prompt,
                config={"temperature": 0.0}
            )
            result = (response.text or "").strip()
            if "None" in result.lower():
                return None
            return result
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower():
                wait = 60 * (attempt + 1)
                print(f"  Rate limit, aguardando {wait}s...")
                time.sleep(wait)
            else:
                print(f"  ERRO: {e}")
                return None
    return None

def get_published_at(doc, max_retries=10):
    # tenta URL primeiro
    date = extract_date_from_url(doc["source_url"])
    if date:
        return date, "url"
    
    # fallback pro LLM com retry ate conseguir
    text = doc.get("text", "")
    if not text:
        return None, None
    
    for attempt in range(max_retries):
        try:
            date = extract_date_from_text(text)
            if date:
                return date, "llm"
            # modelo respondeu mas sem data: não é erro, apenas não encontrou data
            return None, "llm_sem_data"
        except Exception as e:
            wait = 10 * (attempt + 1)
            print(f"  Tentativa {attempt+1}/{max_retries} falhou. Aguardando {wait}s...")
            time.sleep(wait)
    
    # esgotou tentativas, não salva nada
    raise Exception(f"Rate limit persistente após {max_retries} tentativas: {doc['source_url']}")

# HTML Parser

In [3]:
def parse_html_page(url, headers):
    try:
        response = requests.get(url, timeout=10, headers=headers)
        response.raise_for_status()
    except Exception as e:
        print(f"  ERRO: {e}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    # extrai título
    title_tag = soup.find("title")
    title = title_tag.get_text(strip=True) if title_tag else ""

    # pega o conteúdo principal diretamente, ignorando sidebar
    main = soup.find("main", role="main")
    if not main:
        return None

    # remove ruídos dentro do main
    for tag in main.find_all("div", class_="ultimos-posts"):
        tag.decompose()
    for tag in main.find_all("ul", class_="crunchify-social"):
        tag.decompose()
    for tag in main.find_all("a", class_="sr-only"):
        tag.decompose()

    text = main.get_text(separator="\n", strip=True)

    return {
        "source_url": url,
        "title": title,
        "text": text
    }

In [ ]:
# Parsing de HTMLs coletados no crawler
parsed_path = Path("../data/parsed/pages_parsed.json")

if not REPARSE and parsed_path.exists():
    with open(parsed_path, "r", encoding="utf-8") as f:
        results = json.load(f)
    already_parsed = {r["source_url"] for r in results}
    print(f"Já parseadas: {len(results)} páginas")
else:
    results = []
    already_parsed = set()

errors = []

# roda loop de parsing dos htmls
for i, url in enumerate(pages_found):
    # pula URLs já parseadas
    if url in already_parsed:
        continue

    print(f"[{i+1}/{len(pages_found)}] {url}")

    result = parse_html_page(url, HEADERS)

    if result:
        results.append(result)
    else:
        errors.append(url)

    time.sleep(0.3)

# salva resultado
Path("../data/raw").mkdir(parents=True, exist_ok=True)
with open("../data/parsed/pages_parsed.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nParsed: {len(results)} páginas")
print(f"Erros: {len(errors)}")

Já parseadas: 2029 páginas
[769/3764] https://ifrs.edu.br/canoas/nucleos-de-apoio/nucleo-de-educacao-a-distancia-nead/nead@canoas.ifrs.edu.br
[770/3764] https://ifrs.edu.br/canoas/nucleos-de-apoio/nucleo-de-educacao-a-distancia-nead/nead@canoas.ifrs.edu.br
[771/3764] https://ifrs.edu.br/canoas/ensino/diretoria-de-ensino/profmat@canoas.ifrs.edu.br
[772/3764] https://ifrs.edu.br/canoas/ensino/diretoria-de-ensino/profmat@canoas.ifrs.edu.br

Parsed: 2029 páginas
Erros: 4


In [ ]:
# Busca data de publicação dos HTMLs usando URL e Gemini (gemini-2.5-flash-lite)
SAVE_INTERVAL = 50

with open("../data/parsed/pages_parsed.json", "r", encoding="utf-8") as f:
    pages_parsed = json.load(f)

# processa apenas os sem published_at
sem_data = [p for p in pages_parsed if "published_at" not in p]
print(f"HTMLs sem data: {len(sem_data)} de {len(pages_parsed)}")

for i, doc in enumerate(sem_data):
    try:
        date, source = get_published_at(doc)
        if date:
            doc["published_at"] = date
            doc["date_source"] = source
        print(f"[{i+1}/{len(sem_data)}] {source or 'sem data'} — {date or 'N/A'}")
    except Exception as e:
        print(f"  PAROU: {e}")
        with open("../data/parsed/pages_parsed.json", "w", encoding="utf-8") as f:
            json.dump(pages_parsed, f, ensure_ascii=False, indent=2)
        break
    
    # salva incrementalmente
    if (i + 1) % SAVE_INTERVAL == 0:
        with open("../data/parsed/pages_parsed.json", "w", encoding="utf-8") as f:
            json.dump(pages_parsed, f, ensure_ascii=False, indent=2)
        print(f"  Checkpoint salvo: {i+1} processados")

# salva final
with open("../data/parsed/pages_parsed.json", "w", encoding="utf-8") as f:
    json.dump(pages_parsed, f, ensure_ascii=False, indent=2)

print("Enriquecimento de HTMLs concluído.")

# PDF Parser

In [ ]:
# tratamento de pdfs de grades de horários
def is_schedule_pdf(title):
    return "Horários_" in title or "Horarios_" in title

def structure_schedule_text(text):
    prompt = f"""Extraia as informações de professor e disciplina deste horário em frases simples. Adicione o ano primeiro
                Siga EXATAMENTE este formato, uma frase por linha, sem texto adicional:

                Ano documento: 2026
                Professor X leciona Disciplina Y na Sala Z no Curso W semestre N.

                Exemplo:
                Ano documento: 2026
                Rafael Pinto leciona Estrutura de Dados no LAB E10 (INF) no TADS 3º semestre.
                Márcio Bigolin leciona Desenvolvimento Web II no LAB D10 (INF) no TADS 5º semestre.

                Não escreva nada além das frases no formato acima.

                {text}"""

    for attempt in range(3):
        try:
            response = google_client.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=prompt,
                config={"temperature": 0.6}
            )
            content = response.text
            if content is None:
                return text
            return content.strip()
        except Exception as e:
            wait = 30 * (attempt + 1)
            print(f"  ERRO estruturação (tentativa {attempt+1}/3): {e}")
            print(f"  Aguardando {wait}s...")
            time.sleep(wait)
    
    return text

In [7]:
MIN_CHARS = 150

def is_drive_url(url):
    return "drive.google.com" in url

def download_pdf_bytes(url, headers):
    # baixa PDF do Drive usando gdown (salva em arquivo temporário)
    if is_drive_url(url):
        with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
            tmp_path = tmp.name
        try:
            gdown.download(url, tmp_path, quiet=True)
            with open(tmp_path, "rb") as f:
                return f.read()
        except Exception as e:
            print(f"  ERRO gdown: {e}")
            return None
        finally:
            os.remove(tmp_path)
    
    # baixa PDF normal via requests
    try:
        response = requests.get(url, timeout=30, headers=headers)
        response.raise_for_status()
        return response.content
    except Exception as e:
        print(f"  ERRO download: {e}")
        return None

def parse_pdf(pdf_info, headers):
    url = pdf_info["url"]

    # baixa o conteúdo do PDF
    content = download_pdf_bytes(url, headers)
    if content is None:
        return None
    
    if not content.startswith(b"%PDF"):
        print(f"  NÃO É PDF: {url}")
        return {"source_url": url, "format_error": True}

    try:
        # abre o PDF a partir dos bytes
        doc = fitz.open(stream=content, filetype="pdf")

        # extrai título dos metadados
        title = doc.metadata.get("title", "").strip()

        # extrai texto de todas as páginas
        text = ""
        for page in doc:
            text += page.get_text()

        doc.close()
    except Exception as e:
        print(f"  ERRO parse: {e}")
        return None

    # classifica como escaneado se texto for curto demais
    is_scanned = len(text.strip()) < MIN_CHARS

    # reestrutura se for PDF de horário
    if not is_scanned and is_schedule_pdf(title):
        print(f"  Estruturando horário...")
        text = structure_schedule_text(text)
        print(text)

    return {
        "source_url": url,
        "title": title,
        "text": text.strip() if not is_scanned else "",
        "is_scanned": is_scanned,
        "size_kb": pdf_info["size_kb"]
    }

In [ ]:
# Download + parsing de PDFs
pdf_parsed_path = Path("../data/parsed/pdfs_parsed.json")

if not REPARSE and pdf_parsed_path.exists():
    with open(pdf_parsed_path, "r", encoding="utf-8") as f:
        pdf_results = json.load(f)
    already_parsed_pdfs = {r["source_url"] for r in pdf_results}
    print(f"Já parseados: {len(pdf_results)} PDFs")
else:
    pdf_results = []
    already_parsed_pdfs = set()

pdf_errors = []

# roda loop de parsing dos PDFs
for i, pdf_info in enumerate(pdfs_found):
    url = pdf_info["url"]

    # pula URLs já parseadas ou com erro de formato conhecido
    if url in already_parsed_pdfs or url in format_errors:
        continue

    print(f"[{i+1}/{len(pdfs_found)}] {url}")

    result = parse_pdf(pdf_info, HEADERS)

    if result:
        if result.get("format_error"):
            format_errors.add(url)
        else:
            pdf_results.append(result)
    else:
        pdf_errors.append(url)

# salva resultado
Path("../data/parsed").mkdir(parents=True, exist_ok=True)
with open("../data/parsed/pdfs_parsed.json", "w", encoding="utf-8") as f:
    json.dump(pdf_results, f, ensure_ascii=False, indent=2)

# salva erros
with open(format_errors_path, "w", encoding="utf-8") as f:
    json.dump(list(format_errors), f, ensure_ascii=False, indent=2)

print(f"\nParsed: {len(pdf_results)} PDFs")
print(f"Erros: {len(pdf_errors)}")
print(f"Escaneados: {sum(1 for r in pdf_results if r['is_scanned'])}")

Já parseados: 584 PDFs
[28/631] https://drive.google.com/uc?export=download&id=1mffrGzP-HOVkFs2yDnNmQ3CNOv9zUm_2


c:\Users\User\Documents\.dev\IFRS\rag\.venv\Lib\site-packages\gdown\download.py:37: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = bs4.BeautifulSoup(line, features="html.parser")


  ERRO gdown: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1mffrGzP-HOVkFs2yDnNmQ3CNOv9zUm_2

but Gdown can't. Please check connections and permissions.
[29/631] https://drive.google.com/uc?export=download&id=1aYF_S5jN1xnVlwqWYbKk83HBLmcwvWaH
  ERRO gdown: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1aYF_S5jN1xnVlwqWYbKk83HBLmcwvWaH

but Gdown can't. Please check connections and permissions.
[31/631] https://drive.goo

In [ ]:
# Busca data de publicação dos PDFs usando URL e Gemini (gemini-2.5-flash-lite)
with open("../data/parsed/pdfs_parsed.json", "r", encoding="utf-8") as f:
    pdfs_parsed = json.load(f)

sem_data_pdf = [p for p in pdfs_parsed if "published_at" not in p and not p.get("is_scanned")]
print(f"PDFs sem data: {len(sem_data_pdf)} de {len(pdfs_parsed)}")

for i, doc in enumerate(sem_data_pdf):
    date, source = get_published_at(doc)
    doc["published_at"] = date
    doc["date_source"] = source

    print(f"[{i+1}/{len(sem_data_pdf)}] {source or 'sem data'} — {date or 'N/A'}")

    if (i + 1) % SAVE_INTERVAL == 0:
        with open("../data/parsed/pdfs_parsed.json", "w", encoding="utf-8") as f:
            json.dump(pdfs_parsed, f, ensure_ascii=False, indent=2)
        print(f"  Checkpoint salvo: {i+1} processados")

with open("../data/parsed/pdfs_parsed.json", "w", encoding="utf-8") as f:
    json.dump(pdfs_parsed, f, ensure_ascii=False, indent=2)

print("Enriquecimento de PDFs concluído.")